# 11 â€” Geography feature family and robustness

This notebook presents the bounded fixed-model geography screen, fold-fitted hierarchical coordinate imputation and LGA-disjoint sensitivity. It performs no model refits: run `../scripts/run_geography_screen.py` first to recreate the ignored runtime evidence. The labelled local test, competition predictions and oversampling artefacts remain outside this analysis.

In [1]:
from pathlib import Path
import sys

import pandas as pd

STAGE_DIR = Path.cwd().parent
PROJECT_DIR = STAGE_DIR.parent
SRC_DIR = STAGE_DIR / 'src'
RUNTIME_DIR = PROJECT_DIR / '.runtime' / 'geography-screen'
DATA_DIR = STAGE_DIR / 'data'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data_partitioning import make_cross_validation, partition_modelling_data
from feature_engineering import valid_tanzania_coordinates
from geography_evaluation import make_lga_grouped_partition
from geography_features import GEOGRAPHY_POLICIES, GeographyFeatureEngineer
from modelling_data import prepare_modelling_data

required = [
    RUNTIME_DIR / 'frozen-summary.csv',
    RUNTIME_DIR / 'hybrid-summary.csv',
    RUNTIME_DIR / 'lga-grouped-summary.csv',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f'Run the geography screen first; missing={missing!r}')

print(f'Runtime evidence: {RUNTIME_DIR}')

Runtime evidence: C:\_Source\Imperial-ML-AI-Live\Capstone\imperial-capstone\.runtime\geography-screen


## Paired coordinate validity

The source sentinel is `(0, -2e-08)`, so validity uses a paired Tanzania envelope rather than an exact-origin or longitude-only check. Legitimate northern observations near latitude âˆ’1Â° remain valid because their longitudes are around 31Â°.

In [2]:
coordinate_rows = []
for frame_name, filename in (
    ('labelled original', 'TrainingSetValues.csv'),
    ('competition', 'TestSetValues.csv'),
):
    frame = pd.read_csv(DATA_DIR / filename)
    valid = valid_tanzania_coordinates(frame['longitude'], frame['latitude'])
    coordinate_rows.append({
        'frame': frame_name,
        'rows': len(frame),
        'plausible_coordinates': int(valid.sum()),
        'implausible_or_missing': int((~valid).sum()),
        'sentinel_(0,-2e-08)': int(
            frame['longitude'].eq(0).mul(frame['latitude'].eq(-2e-08)).sum()
        ),
    })
pd.DataFrame(coordinate_rows).set_index('frame')

,rows,plausible_coordinates,implausible_or_missing,"sentinel_(0,-2e-08)"
frame,,,,
labelled original,59400,57588,1812,1812
competition,14850,14393,457,457


## Frozen-fold feature-family screen

Every policy uses the unchanged 55% child-weight-1 depth-8 XGBoost and 45% Random Forest vote. The primary gate requires at least +0.10 percentage points, three fold wins, no fold below âˆ’0.25 points and no repair-recall loss beyond two points.

In [3]:
frozen = pd.read_csv(RUNTIME_DIR / 'frozen-summary.csv').set_index('policy')
frozen_display = frozen.loc[:, [
    'label', 'mean_accuracy', 'accuracy_change', 'fold_wins',
    'worst_fold_change', 'repair_recall',
    'transformed_features_fold_1', 'passes_gate',
]].copy()
for column in ('mean_accuracy', 'accuracy_change', 'worst_fold_change', 'repair_recall'):
    frozen_display[column] = frozen_display[column].map(lambda value: f'{value:.3%}')
frozen_display

,label,mean_accuracy,accuracy_change,fold_wins,worst_fold_change,repair_recall,transformed_features_fold_1,passes_gate
policy,,,,,,,,
baseline,Accepted current geography,81.625%,0.000%,0,0.000%,34.859%,301,False
coordinate_centroids,Fold-fitted hierarchical coordinate centroids,81.566%,-0.059%,1,-0.179%,34.859%,299,False
lga_backoff,"Coordinates plus basin, region and LGA",81.547%,-0.078%,1,-0.210%,34.627%,257,False
coordinate_centroids_with_level,Coordinate centroids plus imputation level,81.538%,-0.086%,0,-0.147%,34.743%,303,False
district_composite,Region-district composite,81.526%,-0.099%,1,-0.210%,34.685%,383,False
grid_10km,Current geography plus ~10 km grid,81.507%,-0.118%,2,-0.284%,34.598%,815,False
coarse_region,Coordinates plus basin and region,81.500%,-0.124%,0,-0.179%,34.280%,133,False
grid_20km,Current geography plus ~20 km grid,81.456%,-0.168%,0,-0.210%,34.251%,844,False
lga_ward_composite,Current geography plus LGA-ward,81.454%,-0.170%,1,-0.421%,34.946%,987,False


Coordinates and named geography are complementary. Deleting coordinates costs 0.734 points; deleting named geography costs 0.299 points. None of the composites, grids or centroid variants passes the primary gate.

## Coordinate-centroid fallback evidence

Centroids are coordinate-wise training medians, fitted independently inside each fold. Missing pairs back off through LGA+ward, LGA, region, basin and global levels. The original missingness flag remains one.

In [4]:
modelling_data = prepare_modelling_data(
    pd.read_csv(DATA_DIR / 'TrainingSetValues.csv'),
    pd.read_csv(DATA_DIR / 'TrainingSetLabels.csv'),
    pd.read_csv(DATA_DIR / 'TestSetValues.csv'),
)
partitioned = partition_modelling_data(modelling_data)
policy = GEOGRAPHY_POLICIES['coordinate_centroids_with_level']

def fallback_counts(name, current_partition, cross_validation):
    rows = []
    for fold, (training_positions, validation_positions) in enumerate(
        cross_validation.split(), start=1
    ):
        transformer = GeographyFeatureEngineer(policy).fit(
            current_partition.X_development.iloc[training_positions]
        )
        transformed = transformer.transform(
            current_partition.X_development.iloc[validation_positions]
        )
        missing_rows = transformed.loc[
            transformed['coordinates_missing'].eq(1),
            'coordinate_imputation_level',
        ]
        counts = missing_rows.value_counts()
        rows.append({
            'design': name, 'fold': fold, 'missing_rows': len(missing_rows),
            **{level: int(counts.get(level, 0))
               for level in ('ward', 'lga', 'region', 'basin', 'global')},
        })
    return pd.DataFrame(rows)

frozen_fallbacks = fallback_counts(
    'frozen stratified', partitioned, make_cross_validation(partitioned)
)
grouped_partition, grouped_cv, grouped_folds = make_lga_grouped_partition(partitioned)
grouped_fallbacks = fallback_counts('LGA-disjoint', grouped_partition, grouped_cv)
pd.concat([frozen_fallbacks, grouped_fallbacks], ignore_index=True)

,design,fold,missing_rows,ward,lga,region,basin,global
0,frozen stratified,1,290,30,185,75,0,0
1,frozen stratified,2,276,36,171,69,0,0
2,frozen stratified,3,280,37,172,71,0,0
3,frozen stratified,4,308,25,192,91,0,0
4,frozen stratified,5,284,24,176,84,0,0
5,LGA-disjoint,1,0,0,0,0,0,0
6,LGA-disjoint,2,248,0,0,248,0,0
7,LGA-disjoint,3,800,0,0,800,0,0
8,LGA-disjoint,4,390,0,0,390,0,0
9,LGA-disjoint,5,0,0,0,0,0,0


## LGA-disjoint sensitivity and bounded component crossing

In [5]:
grouped = pd.read_csv(RUNTIME_DIR / 'lga-grouped-summary.csv').set_index('policy')
hybrids = pd.read_csv(RUNTIME_DIR / 'hybrid-summary.csv').set_index('hybrid')
display(grouped.loc[:, [
    'label', 'mean_accuracy', 'accuracy_change', 'fold_wins',
    'worst_fold_change', 'repair_recall',
]])
display(grouped_folds)
display(hybrids)

,label,mean_accuracy,accuracy_change,fold_wins,worst_fold_change,repair_recall
policy,,,,,,
coordinate_centroids,Fold-fitted hierarchical coordinate centroids,0.722613,0.001605,4,-0.001898,0.029899
baseline,Accepted current geography,0.721007,0.000000,0,0.000000,0.034971


,rows,lgas,functional_share,repair_share,non_functional_share
validation_fold,,,,,
1,9489,22,0.546844,0.069976,0.383181
2,9528,24,0.532955,0.084278,0.382767
3,9634,27,0.546606,0.073801,0.379593
4,9384,25,0.539429,0.067349,0.393223
5,9485,27,0.549499,0.067897,0.382604


,mean_accuracy,accuracy_change,fold_wins,worst_fold_change,repair_recall,non_functional_recall
hybrid,,,,,,
baseline XGBoost + centroid Random Forest,0.815930,-0.000316,2,-0.001368,0.348299,0.784599
centroid XGBoost + baseline Random Forest,0.815215,-0.001031,1,-0.001789,0.349745,0.783833


## Decision

Retain the accepted current geography for competition accuracy. Hierarchical centroid imputation is the strongest challenger and improves 4/5 LGA-disjoint folds, but it does not pass the primary frozen-fold gate. The approximately 9.5-point accuracy fall and near-disappearance of repair recall under LGA-disjoint validation are the central robustness warning. Stop geography tuning on these folds; move the next fixed-model feature-family loop to conservative `funder` and `installer` treatments.